<a href="https://colab.research.google.com/github/fserracrespi/ir2_Project/blob/mhm-dev/pairs/10-gpl/reproduce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reproduction of GPL.
Based on Kim's starter notebook, with a few changes.

Previous pipeline:
1. Score BGE-small on SciFact with the shared testbed.
2. Fine-tune it with `MultipleNegativesRankingLoss`: in-batch negatives, the InfoNCE loss from the recap.
3. Mine hard negatives with the spring cross-encoder and train again.
4. Measure alignment and uniformity (Wang & Isola 2020).
5. Optional: train a LoRA adapter with `peft` instead of the whole model.

The positive pairs here are a paper's title and its abstract, taken from the corpus. No test query and no relevance label is used for training: `queries` and `qrels` only appear inside `evaluate`.

On Colab: *Runtime → Change runtime type → T4 GPU*, then run all cells.

In [1]:
# changed to access our own testbed. might need changing before
# committing to Kim's branch.
import os
if not os.path.exists("testbed.py"):             # on Colab: fetch the repo
    !git clone -q -b mhm-dev https://github.com/fserracrespi/ir2_Project
    %cd ir2_Project
!pip install -q -r requirements.txt datasets accelerate peft

/content/ir2_Project


Checking we're in the right branch

In [2]:
!git branch --show-current

mhm-dev


In [3]:
import random
import numpy as np
import torch
from datasets import Dataset
from sentence_transformers import (CrossEncoder, SentenceTransformer, SentenceTransformerTrainer,
                                   SentenceTransformerTrainingArguments, losses)
import testbed

SEED = 0
MODEL = "BAAI/bge-small-en-v1.5"
random.seed(SEED)
np.random.seed(SEED)
corpus, queries, qrels = testbed.load("scifact")
N_TRAIN, BATCH = len(corpus) // 2, 32 # (original 2000 train queries)
QUERIES_PER_DOC = 3
N_MINE = 10
# N_TEST = N_TRAIN // 2
# docs = [(d["title"], d["text"]) for d in corpus.values() if d["title"] and len(d["text"].split()) >= 50] # this loads the docs we will train on
# random.shuffle(docs)
# train_docs, held_out = docs[:N_TRAIN], docs[N_TRAIN:N_TRAIN + N_TEST]
# print(len(corpus), "documents,", len(docs), "usable,", len(train_docs), "for training,", len(held_out), "held out")

/tmp/ipykernel_1310/333118757.py:5: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import (CrossEncoder, SentenceTransformer, SentenceTransformerTrainer,
downloading scifact


In [4]:
print(testbed.device())

cuda


## 1. Before training

`evaluate` is the shared testbed: the same search and the same scoring as every row of the results table.

In [5]:
def evaluate(model, label):
    run = testbed.dense_search(model, corpus, queries, query_prefix=testbed.BGE_QUERY)
    _, ndcg, recall = testbed.score_run(run, "scifact")
    print(f"{label:24} nDCG@10 {ndcg.mean():.3f}  Recall@100 {recall.mean():.3f}")
    return ndcg

model = SentenceTransformer(MODEL, device=testbed.device())
before = evaluate(model, "BGE-small, as downloaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

BGE-small, as downloaded nDCG@10 0.710  Recall@100 0.948


# 2 GPL Training of bge-small
## 2.1 T5 query generation. For each document in training, we generate three queries for it

In [7]:
# data preparation. They use title + abstract for all models
print(type(corpus))

doc_ids = list(corpus)
idx = {d: n for n, d in enumerate(doc_ids)}
all_texts = [corpus[d]["title"] + " " + corpus[d]["text"] for d in doc_ids] # encode all texts

print(doc_ids[0])
shuffled = doc_ids[:]
random.shuffle(shuffled)
train_ids = shuffled[:N_TRAIN] # we just work with a couple of documents for now
title_abstract_train = [corpus[doc]["title"] + " " + corpus[doc]["text"] for doc in train_ids]
print(len(train_ids), len(title_abstract_train))
print(train_ids[0], title_abstract_train[0][:150])


<class 'dict'>
4983
2591 2591
4462155 A temporal shift in the circuits mediating retrieval of fear memory Fear memories allow animals to avoid danger, thereby increasing their chances of s


In [8]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
t5_model_name = 'doc2query/msmarco-t5-base-v1'
t5_tokenizer = T5Tokenizer.from_pretrained(t5_model_name)
t5_model = T5ForConditionalGeneration.from_pretrained(t5_model_name).to(testbed.device())




tokenizer_config.json:   0%|          | 0.00/2.12k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [9]:
torch.manual_seed(SEED)
# generation loop
generated = []
for i in range(0, len(title_abstract_train), BATCH):
  ids = train_ids[i: i+BATCH]
  texts = title_abstract_train[i: i+ BATCH]
  # we use max_length 350 bc that is what they use in the paper, but also because this is a reasonable length for title + abstract
  encoded_out = t5_tokenizer(texts, max_length=350, truncation=True, padding=True, return_tensors="pt").to(t5_model.device)
  generation_out = t5_model.generate(**encoded_out, max_length=64, do_sample=True, top_k=25, top_p=0.95, num_return_sequences=QUERIES_PER_DOC)
  decoded = t5_tokenizer.batch_decode(generation_out, skip_special_tokens=True)
  generated += [{"query": q, "pos": ids[k//QUERIES_PER_DOC]} for k, q in enumerate(decoded)]


In [10]:
# save it so we can retrieve later from Drive
import json, os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/gpl", exist_ok=True)
with open("/content/drive/MyDrive/gpl/scifact_queries_half_corpus.json", "w") as f:
    json.dump(generated, f)
print("saved", len(generated), "queries")

Mounted at /content/drive
saved 7773 queries


In [11]:
for i in generated[:3]:
  doc = corpus[i["pos"]]
  query = i["query"]
  print(f"corpus title and abstract: {doc} \n query: {query}")
  print()

corpus title and abstract: {'title': 'A temporal shift in the circuits mediating retrieval of fear memory', 'text': 'Fear memories allow animals to avoid danger, thereby increasing their chances of survival. Fear memories can be retrieved long after learning, but little is known about how retrieval circuits change with time. Here we show that the dorsal midline thalamus of rats is required for the retrieval of auditory conditioned fear at late (24 hours, 7 days, 28 days), but not early (0.5 hours, 6 hours) time points after learning. Consistent with this, the paraventricular nucleus of the thalamus (PVT), a subregion of the dorsal midline thalamus, showed increased c-Fos expression only at late time points, indicating that the PVT is gradually recruited for fear retrieval. Accordingly, the conditioned tone responses of PVT neurons increased with time after training. The prelimbic (PL) prefrontal cortex, which is necessary for fear retrieval, sends dense projections to the PVT. Retrieva

Example 3 queries / passage
corpus title and abstract: {'title': 'A temporal shift in the circuits mediating retrieval of fear memory', 'text': 'Fear memories allow animals to avoid danger, thereby increasing their chances of survival.} ...
 query 1: how do conditioned nerves retrieve fear

 query 2: which neuron is responsible for fear retrieval

 query 3: does the pvt stimulate fear




Example generated queries

corpus title and abstract: {'title': 'Cardiovascular and renal benefits of dry bean and soybean intake.', 'text': 'Dry beans and soybeans are nutrient-dense, fiber-rich, and are high-quality sources of protein. Protective and therapeutic effects of both dry bean and soybean intake have been documented. Studies show that dry bean intake has the potential to decrease serum cholesterol concentrations, improve many aspects of the diabetic state, and provide metabolic benefits that aid in weight control. Soybeans are a unique source of the isoflavones genistein and diadzein, which have numerous biological functions. Soybeans and soyfoods potentially have multifaceted health-promoting effects, including cholesterol reduction, improved vascular health, preserved bone mineral density, and reduction of menopausal symptoms. Soy appears to have salutary effects on renal function, although these effects are not well understood. Whereas populations consuming high intakes of soy have lower prevalences of certain cancers, definitive experimental data are insufficient to clarify a protective role of soy. The availability of legume products and resources is increasing, incorporating dry beans and soyfoods into the diet can be practical and enjoyable. With the shift toward a more plant-based diet, dry beans and soy will be potent tools in the treatment and prevention of chronic disease.'}:
 query: is soy good for the kidneys

corpus title and abstract: {'title': 'The 5TMM series: a useful in vivo mouse model of human multiple myeloma.', 'text': 'The present invention provides a combination sink and dishwashing apparatus having a sink sharing a common side wall with a cabinet which defines a closed space. The cabinet has a wire basket for holding and washing a plurality of dishes within the cabinet. The common side wall is a part of the cabinet that defines the closed space and is positioned to form one side of the sink. The common side wall can be opened to allow the wire basket to slide from within the cabinet and into the sink, for loading and unloading the dishes. Within the cabinet, the invention contains a pump powered by a motor, the pump spraying water through a rotatably mounted spray arm onto the dishes to wash them, as with traditional dishwashing machines. The invention preferably includes a garbage disposal which is also powered by the motor. In an alternative embodiment, this invention can include two cabinets as described above, each located on opposing sides of the sink.'}:
 query: what is a combination sink and dishwashing apparatus

corpus title and abstract: {'title': 'Transient receptor potential channels as drug targets: from the science of basic research to the art of medicine.', 'text': 'The large Trp gene family encodes transient receptor potential (TRP) proteins that form novel cation-selective ion channels. In mammals, 28 Trp channel genes have been identified. TRP proteins exhibit diverse permeation and gating properties and are involved in a plethora of physiologic functions with a strong impact on cellular sensing and signaling pathways. Indeed, mutations in human genes encoding TRP channels, the so-called "TRP channelopathies," are responsible for a number of hereditary diseases that affect the musculoskeletal, cardiovascular, genitourinary, and nervous systems. This review gives an overview of the functional properties of mammalian TRP channels, describes their roles in acquired and hereditary diseases, and discusses their potential as drug targets for therapeutic intervention.'}:
 query: what is the function of the trp gene



In [12]:
# after generation check
print(len(generated) == len(title_abstract_train) * QUERIES_PER_DOC)

True


In [13]:
# retrieve the file instead of running generation again after a disconnect
import json
from google.colab import drive
drive.mount("/content/drive")
with open("/content/drive/MyDrive/gpl/scifact_queries_half_corpus.json") as f:
    generated = json.load(f)
print("loaded", len(generated), "queries")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
loaded 7773 queries


## 2.2 For each of the generated queries, they use a an existing retrieval system to retrieve 50 negative passages.

In specific they use two dense retrievers from Sentence-Transformers:
- msmarco-distilbert-base-v3 and
- msmarco-MiniLML-6-v3

We tried with different amount of negatives to see how ndcg changed. However, they all take about the same time, since

In [14]:
print(title_abstract_train[0])
print(generated[0])

A temporal shift in the circuits mediating retrieval of fear memory Fear memories allow animals to avoid danger, thereby increasing their chances of survival. Fear memories can be retrieved long after learning, but little is known about how retrieval circuits change with time. Here we show that the dorsal midline thalamus of rats is required for the retrieval of auditory conditioned fear at late (24 hours, 7 days, 28 days), but not early (0.5 hours, 6 hours) time points after learning. Consistent with this, the paraventricular nucleus of the thalamus (PVT), a subregion of the dorsal midline thalamus, showed increased c-Fos expression only at late time points, indicating that the PVT is gradually recruited for fear retrieval. Accordingly, the conditioned tone responses of PVT neurons increased with time after training. The prelimbic (PL) prefrontal cortex, which is necessary for fear retrieval, sends dense projections to the PVT. Retrieval at late time points activated PL neurons projec

In [15]:
from sentence_transformers import SentenceTransformer, util
dense_retrieval_distil = SentenceTransformer('sentence-transformers/msmarco-distilbert-base-v3')
# first embed the documents
# we normalize embeddings to calculate cosine similarity
embeddings_title_abstract = dense_retrieval_distil.encode(all_texts, normalize_embeddings=True, batch_size=BATCH, show_progress_bar=True)
# then all the queries
embeddings_queries = dense_retrieval_distil.encode([g["query"] for g in generated], normalize_embeddings=True, batch_size=BATCH, show_progress_bar=True)
top_51_distil = util.semantic_search(embeddings_queries, embeddings_title_abstract, top_k=N_MINE + 1)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.53k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/545 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  265MB            

... (6 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/162 [00:00<?, ?it/s]

Batches:   0%|          | 0/243 [00:00<?, ?it/s]

5183 5183


In [16]:
print(top_51_distil[0])
print(type(top_51_distil))


idx = {d: n for n, d in enumerate(doc_ids)}
neg_distil, pos_distil = [], []
for i, hits in enumerate(top_51_distil):
    pos_id = generated[i]["pos"]
    q = {"query": generated[i]["query"], "ids": [], "texts": []}
    pos_distil.append({"pos_id": pos_id, "text": all_texts[idx[pos_id]]})
    for h in hits:
        doc_id = doc_ids[h["corpus_id"]]
        if doc_id != pos_id:
            q["ids"].append(doc_id)
            q["texts"].append(all_texts[h["corpus_id"]])
    q["ids"], q["texts"] = q["ids"][:N_MINE], q["texts"][:N_MINE]
    neg_distil.append(q)

print(len(neg_distil), len(neg_distil[0]["texts"]))
print(neg_distil[0]["query"])
print(pos_distil[0]["text"][:100])
print(corpus[generated[0]["pos"]]["title"])

[{'corpus_id': 868, 'score': 0.5610849857330322}, {'corpus_id': 2558, 'score': 0.3991600275039673}, {'corpus_id': 5059, 'score': 0.3960225582122803}, {'corpus_id': 759, 'score': 0.3904815912246704}, {'corpus_id': 4682, 'score': 0.3752187192440033}, {'corpus_id': 3488, 'score': 0.3751799166202545}, {'corpus_id': 1783, 'score': 0.37273508310317993}, {'corpus_id': 3666, 'score': 0.37182343006134033}, {'corpus_id': 4429, 'score': 0.36701202392578125}, {'corpus_id': 20, 'score': 0.3602646589279175}, {'corpus_id': 4606, 'score': 0.35929495096206665}]
<class 'list'>
7773 10
how do conditioned nerves retrieve fear
A temporal shift in the circuits mediating retrieval of fear memory Fear memories allow animals to a
A temporal shift in the circuits mediating retrieval of fear memory


In [17]:
i = 9
g = generated[i]

print("QUERY:   ", g["query"])
print("POSITIVE:", corpus[g["pos"]]["title"], "\n")
for h in top_51_distil[i][:4]:
    d = doc_ids[h["corpus_id"]]
    mark = "  <- positive" if d == g["pos"] else ""
    print(f"{h['score']:.3f}  {corpus[d]['title']}{mark}")

QUERY:    what is the cause of aging in a male
POSITIVE: Inducible depletion of satellite cells in adult, sedentary mice impairs muscle regenerative capacity without affecting sarcopenia 

0.467  The rate of change in declining steroid hormones: a new parameter of healthy aging in men?
0.399  Psychobiological Protective Factors Modifying the Association Between Age and Sexual Health in Men: Findings From the Men’s Health 40+ Study
0.386  Why females live longer than males: control of longevity by sex hormones.
0.377  Sex-specific effects of the DAF-12 steroid receptor on aging in Caenorhabditis elegans.


In [18]:
# now the same but with the second dense retrieval
all_texts = [corpus[d]["title"] + " " + corpus[d]["text"] for d in doc_ids] # encode all texts
dense_retrieval_minilm = SentenceTransformer('sentence-transformers/msmarco-MiniLM-L6-v3')
# first embed the documents
# we normalize embeddings to calculate cosine similarity
embeddings_title_abstract_minilm = dense_retrieval_minilm.encode(all_texts, normalize_embeddings=True, batch_size=BATCH, show_progress_bar=True)
# then all the queries
embeddings_queries_minilm = dense_retrieval_minilm.encode([g["query"] for g in generated], normalize_embeddings=True, batch_size=BATCH, show_progress_bar=True)
top_51_minilm = util.semantic_search(embeddings_queries_minilm, embeddings_title_abstract_minilm, top_k=N_MINE + 1)
print(len(embeddings_title_abstract_minilm), len(doc_ids))   # both 5183

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.50k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (6 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/162 [00:00<?, ?it/s]

Batches:   0%|          | 0/243 [00:00<?, ?it/s]

5183 5183


In [19]:
i = 9
g = generated[i]
print("QUERY:   ", g["query"])
print("POSITIVE:", corpus[g["pos"]]["title"], "\n")

for h in top_51_minilm[i][:4]:
    d = doc_ids[h["corpus_id"]]
    mark = "  <- positive" if d == g["pos"] else ""
    print(f"{h['score']:.3f}  {corpus[d]['title']}{mark}")

QUERY:    what is the cause of aging in a male
POSITIVE: Inducible depletion of satellite cells in adult, sedentary mice impairs muscle regenerative capacity without affecting sarcopenia 

0.506  The rate of change in declining steroid hormones: a new parameter of healthy aging in men?
0.443  Psychobiological Protective Factors Modifying the Association Between Age and Sexual Health in Men: Findings From the Men’s Health 40+ Study
0.440  Sex-specific effects of the DAF-12 steroid receptor on aging in Caenorhabditis elegans.
0.430  The intersection between aging and cardiovascular disease.


In [20]:
# same for minilm negatives
neg_minilm = []
for i, hits in enumerate(top_51_minilm):
    pos_id = generated[i]["pos"]
    q = {"query": generated[i]["query"], "ids": [], "texts": []}
    for h in hits:
        doc_id = doc_ids[h["corpus_id"]]
        if doc_id != pos_id:
            q["ids"].append(doc_id)
            q["texts"].append(all_texts[h["corpus_id"]])
    q["ids"], q["texts"] = q["ids"][:N_MINE], q["texts"][:N_MINE]
    neg_minilm.append(q)

print(len(neg_minilm), len(neg_minilm[0]["texts"]))
print(neg_minilm[0]["query"])
print(corpus[generated[0]["pos"]]["title"])

7773 10
how do conditioned nerves retrieve fear
A temporal shift in the circuits mediating retrieval of fear memory


In [21]:
# now we put the results of the miners together for downstream processing: cross-encoder pseudo labeling

pool_ids = []
for a, b in zip(neg_distil, neg_minilm):
  # this creates a list of the unique dictionary keys that would be built from a and b
  pool_ids.append(list(dict.fromkeys(a["ids"] + b["ids"])))

for i in range(3,13):
  print(len(pool_ids[i]))


19
20
19
19
20
20
12
17
20
15


We can see both miners retrieve different passages for the same query. However, they both rank first the positive query, which is a sign they are both able to retrieve the most relevant passage. It is interesting to have both miners, since they encode the information differently.

In [22]:
# save it so we don't have to run all models again
mined = {"generated": generated, "pool_ids": pool_ids}
with open("/content/drive/MyDrive/gpl/scifact_mined_20.json", "w") as f:
  json.dump(mined, f)
print("saved", len(mined["generated"]), "queries and", len(mined["pool_ids"]), "pools")

saved 7773 queries and 7773 pools


In [25]:
# and to load the queries and pools
import json
with open("/content/drive/MyDrive/gpl/scifact_mined_20.json") as f:
  mined = json.load(f)
generated, pool_ids = mined["generated"], mined["pool_ids"]

print("loaded", len(generated), "queries and", len(pool_ids), "pools")

loaded 7773 queries and 7773 pools


## 2.3 Cross-encoder pseudo labeling
They do 140k training steps and use 32 batch size

In [26]:
from sentence_transformers import CrossEncoder

ce = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')
# this is how many negatives per generated query we want to take from the pool
N_NEGS_QUERY = 1

# random choice from pool to get the texts that will be encoded with the query in this round

neg_texts_chosen = []
for query_negs in pool_ids:
  neg_texts_chosen.append(all_texts[idx[np.random.choice(query_negs)]])

query_texts = []
posi_texts = []

for g in generated:
  query_texts.append(g["query"])
  posi_texts.append(all_texts[idx[g["pos"]]])


neg_texts_chosen = []
for i in range(N_NEGS_QUERY):
  negs_i = []
  for query_negs in pool_ids:
    negs_i.append(all_texts[idx[np.random.choice(query_negs)]])
  neg_texts_chosen.append(negs_i)

posi_ce_scores = ce.predict(list(zip(query_texts, posi_texts)), batch_size=BATCH, show_progress_bar=True)
margins = []

for i in range(N_NEGS_QUERY):
  neg_training_i = neg_texts_chosen[i]
  neg_ce_scores = ce.predict(list(zip(query_texts, neg_training_i)), batch_size=BATCH, show_progress_bar=True)
  margins_i = posi_ce_scores - neg_ce_scores
  margins.append(margins_i)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Batches:   0%|          | 0/243 [00:00<?, ?it/s]

Batches:   0%|          | 0/243 [00:00<?, ?it/s]

In [27]:
print(neg_ce_scores)

[-11.016296   -5.9732375  -5.7515373 ...   1.4486623   0.6907812
 -10.842368 ]


We can see that most positive texts get a positive score, meaning the cross-encoder thinks the positive passage is helpful for answering the query. Some get negative score, which could be due to the previous problem we spotted: some abstracts are not matched with their title.
We can also see most negative passages get a negative score. Training our model with these soft labels will be more helpful than binary 1/0 as QGen does, and it will allow it to learn even if the texts in the database are not well paired with their titles. And also even if the query is badly generated, thanks to the cross encoder being able to assign a low value to the query-passage + pair.

# 2.4 Build examples for training
They build the synthetic dataset {(Qi, Pi, Pi-, CE_margin)}

In [28]:
# build examples
examples = []
for i in range(N_NEGS_QUERY):
  for j in range(len(query_texts)):
    examples.append((query_texts[j], posi_texts[j], neg_texts_chosen[i][j], float(margins[i][j])))

In [29]:
# save synthetic dataset
dataset = {"generated": generated, "pool_ids": pool_ids, "examples": examples}
with open("/content/drive/MyDrive/gpl/scifact_synthetic_dataset_half_3_10.json", "w") as f:
  json.dump(dataset, f)
print("saved", len(dataset["generated"]), "queries, ", len(dataset["pool_ids"]), "pools and", len(dataset["examples"]), "examples.")


saved 7773 queries,  7773 pools and 7773 examples.


In [33]:
# and to load the examples
import json
with open("/content/drive/MyDrive/gpl/scifact_synthetic_dataset_half_3_10.json") as f:
  dataset = json.load(f)
generated, pool_ids, examples = dataset["generated"], dataset["pool_ids"], dataset["examples"]
print("loaded", len(generated), "queries, ", len(pool_ids), "pools and", len(examples), "examples.")


loaded 7773 queries,  7773 pools and 7773 examples.


## 2.5 Train the student
They used:
- BAAI/bge-small-en-v1.5
- sentence-transformers/all-MiniLM-L6-v2

Loss:
marginMSE loss (Hofstatter et al 2020)


In [34]:
def train(model, data, output_dir, lr=2e-5):
    args = SentenceTransformerTrainingArguments(output_dir=output_dir, num_train_epochs=1, per_device_train_batch_size=BATCH,
                                                learning_rate=lr, logging_steps=20, save_strategy="no", report_to="none", seed=SEED,
                                                use_cpu=not torch.cuda.is_available())
    loss = losses.MarginMSELoss(model)
    SentenceTransformerTrainer(model=model, args=args, train_dataset=data, loss=loss).train()

In [35]:
train_dataset = Dataset.from_dict({
    "text1": [testbed.BGE_QUERY + q[0] for q in examples],
    "text2": [q[1] for q in examples],
    "text3": [q[2] for q in examples],
    "label": [q[3] for q in examples],
})

student = SentenceTransformer(MODEL, device=testbed.device())
train(student, train_dataset, "runs/starter")
after = evaluate(student, "after in-batch training")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
20,119.063428
40,118.386267
60,110.668896
80,117.538843
100,116.953198
120,113.924487
140,114.248853
160,112.785925
180,112.899036
200,115.159778


after in-batch training  nDCG@10 0.424  Recall@100 0.757


In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Would remove:
    /usr/local/lib/python3.13/dist-packages/torchao-0.10.0.dist-info/*
    /usr/local/lib/python3.13/dist-packages/torchao/*
Proceed (Y/n)? Y
  Successfully uninstalled torchao-0.10.0


In [ ]:
USE_LORA = True
if USE_LORA:
    from peft import LoraConfig, TaskType
    lora_model = SentenceTransformer(MODEL, device=testbed.device())
    lora_model.add_adapter(LoraConfig(task_type=TaskType.FEATURE_EXTRACTION, inference_mode=False,
                                      r=8, lora_alpha=16, lora_dropout=0.1))
    train(lora_model, train_dataset, "runs/starter", lr=1e-4)
    lora = evaluate(lora_model, "after LoRA training")
    print("LoRA vs original:", testbed.compare(lora, before))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
20,86.534979
40,81.631616


after LoRA training      nDCG@10 0.713  Recall@100 0.952
LoRA vs original: {'delta': 0.0029, 'p': 0.3045928465631482, 'ci95': [-0.0026, 0.0085]}


In [ ]:
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"trainable: {trainable:,} of {total:,} ({100 * trainable / total:.2f}%)")

count_params(student)
count_params(lora_model)

trainable: 33,360,000 of 33,360,000 (100.00%)
trainable: 147,456 of 33,507,456 (0.44%)


In [8]:
testbed.BGE_QUERY

'Represent this sentence for searching relevant passages: '

## 2. Alignment and uniformity

Alignment: how close the two views of a paper are. Uniformity: how evenly the papers spread over the sphere. Both are better when lower. These are the ten lines of NumPy, on title/abstract pairs the model does not train on.

In [ ]:
def alignment_uniformity(model, pairs):
    a = model.encode([testbed.BGE_QUERY + t for t, _ in pairs], normalize_embeddings=True, batch_size=64)
    b = model.encode([x for _, x in pairs], normalize_embeddings=True, batch_size=64)
    alignment = ((a - b) ** 2).sum(1).mean()
    squared = ((b[:, None] - b[None]) ** 2).sum(-1)[np.triu_indices(len(b), 1)]
    uniformity = np.log(np.exp(-2 * squared).mean())
    return round(float(alignment), 3), round(float(uniformity), 3)

print("alignment, uniformity:", alignment_uniformity(model, held_out))

## 3. Contrastive training with in-batch negatives

Each title must find its own abstract among the abstracts of the batch: with a batch of 32, one positive and 31 negatives. The anchor gets BGE's query prefix, because at test time the query does.

In [ ]:
def train(model, data, output_dir):
    args = SentenceTransformerTrainingArguments(output_dir=output_dir, num_train_epochs=1, per_device_train_batch_size=BATCH,
                                                learning_rate=2e-5, logging_steps=20, save_strategy="no", report_to="none", seed=0,
                                                use_cpu=not torch.cuda.is_available())
    loss = losses.MultipleNegativesRankingLoss(model)
    SentenceTransformerTrainer(model=model, args=args, train_dataset=data, loss=loss).train()

pairs = Dataset.from_dict({"anchor": [testbed.BGE_QUERY + t for t, _ in train_docs], "positive": [x for _, x in train_docs]})
train(model, pairs, "runs/starter")
after = evaluate(model, "after in-batch training")
print("alignment, uniformity:", alignment_uniformity(model, held_out))

## 4. Hard negatives from the spring cross-encoder

Random abstracts are easy to reject. So for each title we take the current model's top 20 abstracts, ask the cross-encoder which wrong one looks most relevant, and add it to the loss as an extra negative.

In [ ]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=testbed.device())
abstracts = [x for _, x in train_docs]
D = model.encode(abstracts, normalize_embeddings=True, batch_size=64)
triples = {"anchor": [], "positive": [], "negative": []}
for i, (title, abstract) in enumerate(train_docs[:N_MINE]):
    q = model.encode(testbed.BGE_QUERY + title, normalize_embeddings=True)
    wrong = [j for j in np.argsort(-(D @ q))[:21] if j != i][:20]
    scores = cross_encoder.predict([[title, abstracts[j]] for j in wrong], show_progress_bar=False)
    triples["anchor"].append(testbed.BGE_QUERY + title)
    triples["positive"].append(abstract)
    triples["negative"].append(abstracts[wrong[int(np.argmax(scores))]])

i = 0
print("title:        ", train_docs[i][0])
print("hard negative:", triples["negative"][i][:160], "…")
train(model, Dataset.from_dict(triples), "runs/starter")
hard = evaluate(model, "after hard negatives")

## 5. Did it change anything?

The same test as in the results table: a paired t-test and a bootstrap interval over the 300 queries. A small difference with a large p-value is noise, not a win.

In [ ]:
print("in-batch vs original:      ", testbed.compare(after, before))
print("hard negatives vs original:", testbed.compare(hard, before))

## 6. Optional: a LoRA adapter

Instead of updating all the weights, train a small adapter on top of frozen ones. Set `USE_LORA = True`. Tested with BGE-small; another architecture may need `target_modules` in `LoraConfig`.

In [ ]:
USE_LORA = False
if USE_LORA:
    from peft import LoraConfig, TaskType
    lora_model = SentenceTransformer(MODEL, device=testbed.device())
    lora_model.add_adapter(LoraConfig(task_type=TaskType.FEATURE_EXTRACTION, inference_mode=False,
                                      r=8, lora_alpha=16, lora_dropout=0.1))
    train(lora_model, pairs, "runs/starter")
    lora = evaluate(lora_model, "after LoRA training")
    print("LoRA vs original:", testbed.compare(lora, before))

## From here to your testbed row

1. Keep your model: `model.save("runs/my-model")`.
2. Start from `pairs/_template/method.py`: it writes one run per dataset to `runs/`.
3. Score it: `python testbed.py score pairs/<your folder>/runs --method "<method>" --pair <your folder> --against <closest baseline>`.